In [24]:
"""Render the 'What Covers Bangladesh' map. Usage:
python 2_render_map.py bangladesh_lulc_2025_200m.tif --year 2025 --merge-flooded none
--merge-flooded: none (keep 7th class) | crops | water | trees
"""
import argparse, numpy as np, rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from scipy.ndimage import gaussian_filter
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pyproj import Transformer
from PIL import Image


ap = argparse.ArgumentParser()
ap.add_argument("tif", nargs="?", default=r"C:\Users\Juvair\Desktop\DataViz_Linkedin\withjuv_bd\bangladesh_lulc_2025_200m.tif")
ap.add_argument("--year", default="2025")
ap.add_argument("--merge-flooded", default="none", choices=["none", "crops", "water", "trees"])
ap.add_argument("--out", default=r"C:\Users\Juvair\Desktop\DataViz_Linkedin\withjuv_bd\what_covers_bangladesh_v11.png")
a, _ = ap.parse_known_args([])

NAMES = {1: "Crops", 2: "Rangeland", 3: "Trees", 4: "Built area", 5: "Bare ground", 6: "Water", 7: "Flooded veg."}
COL = {1: "#76B041", 2: "#CDB885", 3: "#1B4D33", 4: "#c9c9bf", 5: "#EADFC4", 6: "#2F80C9", 7: "#6FB7A8"}
GREEN, BG, INK = "#2E8B45", "#E0E0E0", "#14131A"

# ---- load, equal-area-ish grid (UTM 46N, 200 m pixels => pixel count = area)
with rasterio.open(a.tif) as src:
    if src.crs.to_epsg() == 32646:
        arr, tr = src.read(1), src.transform
    else:
        tr, w, h = calculate_default_transform(src.crs, "EPSG:32646", src.width, src.height, *src.bounds, resolution=200)
        arr = np.zeros((h, w), np.uint8)
        reproject(rasterio.band(src, 1), arr, dst_transform=tr, dst_crs="EPSG:32646",
                  resampling=Resampling.nearest, dst_nodata=0)
if a.merge_flooded != "none":
    arr[arr == 7] = {"crops": 1, "water": 6, "trees": 3}[a.merge_flooded]
rows, cols = np.where(arr > 0)
r0, r1, c0, c1 = rows.min(), rows.max() + 1, cols.min(), cols.max() + 1
arr = arr[r0:r1, c0:c1]
left = tr.c + c0 * tr.a; top = tr.f + r0 * tr.e
right = left + arr.shape[1] * tr.a; bottom = top + arr.shape[0] * tr.e
ext = (left, right, bottom, top)
land = arr > 0
shares = {k: (arr == k).sum() / land.sum() * 100 for k in NAMES if (arr == k).any()}

# ---- figure
fig = plt.figure(figsize=(8.19, 10.24), dpi=200, facecolor=BG)
fig.text(0.04, 0.955, "BANGLADESH  ·  LAND", color=GREEN, fontsize=13, fontweight="bold")
fig.text(0.04, 0.905, "What Covers Bangladesh", color=INK, fontsize=30, fontweight="bold")
fig.text(0.04, 0.873, f"Land cover from 10-metre satellite imagery, {a.year}", color="#555", fontsize=14)
#fig.text(0.96, 0.955, "WithJuv", color=GREEN, fontsize=22, fontweight="bold", ha="right")

logo = Image.open(r"C:\Users\Juvair\Desktop\DataViz_Linkedin\withjuv_logo_white.png").convert("RGBA")
logo_ax = fig.add_axes([0.75, 0.87, 0.24, 0.11])
#([0.82, 0.925, 0.14, 0.045])
logo_ax.imshow(logo)
logo_ax.axis("off")

ax = fig.add_axes([0.03, 0.215, 0.94, 0.63]); ax.axis("off")
W = right - left; depth = 0.014 * W; n = 28
#pad = 0.12 * W
pad = 0.0085 * W
ax.set_xlim(left - pad, right + pad); ax.set_ylim(bottom - pad, top + pad * 0.6); ax.set_aspect("equal")

# # ---- Main map zoom control
# zoom = 1.15  # Increase to zoom in; decrease to zoom out

# W = right - left
# H = top - bottom

# cx = (left + right) / 2
# cy = (bottom + top) / 2

# ax.set_xlim(cx - W / (2 * zoom), cx + W / (2 * zoom))
# ax.set_ylim(cy - H / (2 * zoom), cy + H / (2 * zoom))
# ax.set_aspect("equal")


blur = gaussian_filter(land.astype(np.float32), 6) * 0.28
sh = np.zeros(arr.shape + (4,), np.uint8); sh[..., 3] = (blur * 255).astype(np.uint8)
ax.imshow(sh, extent=(ext[0] + depth*1.3, ext[1] + depth*1.3, ext[2] - depth*2.2, ext[3] - depth*2.2), zorder=1)
side = np.zeros(arr.shape + (4,), np.uint8); side[land] = (138, 117, 87, 255)
for k in range(n, 0, -1):
    d = depth * k / n
    ax.imshow(side, extent=(ext[0] + d*0.35, ext[1] + d*0.35, ext[2] - d, ext[3] - d), zorder=2)
rgba = np.zeros(arr.shape + (4,), np.uint8)
for k, c in COL.items():
    rgba[arr == k] = [int(v * 255) for v in matplotlib.colors.to_rgba(c)]
ax.imshow(rgba, extent=ext, zorder=3, interpolation="nearest")

# ---- labels (lon, lat, label dx km, dy km, ha)
T = Transformer.from_crs(4326, 32646, always_xy=True)
places = [("Dhaka", 90.41, 23.81,190,30, "left"), ("Chattogram", 91.78, 22.34, 95, -5, "left"),
          ("Khulna", 89.57, 22.82, -105, -25, "right"), ("Sylhet", 91.87, 24.90, 70, 55, "left"),
          ("Rajshahi", 88.60, 24.37, -70, 40, "right"), ("Cox's Bazar", 91.98, 21.45, 75, -45, "left"),
          ("Sundarbans", 89.25, 22.0, -85, -75, "right"),("Rangpur", 89.27, 25.74, -80, 50, "right")]
for name, lon, lat, dx, dy, ha in places:
    x, y = T.transform(lon, lat)
    ax.plot(x, y, "o", color="#111", ms=4.5, zorder=6)
    ax.annotate(name, (x, y), (x + dx*1000, y + dy*1000), ha=ha, va="center", fontsize=13.5,
                fontweight="bold", color=INK, zorder=7,
                arrowprops=dict(arrowstyle="-", color="#222", lw=0.9, connectionstyle="arc3,rad=0.25",
                                shrinkA=3, shrinkB=3))

# ---- stat + legend
top_k = max(shares, key=shares.get)
fig.text(0.04, 0.105, f"{shares[1]:.0f}%", color=GREEN, fontsize=62, fontweight="bold")
fig.text(0.04, 0.062, f"of Bangladesh is cropland;\nforest covers {shares.get(3, 0):.0f}%", color=INK,
         fontsize=13, fontweight="bold", linespacing=1.3)
fig.text(0.41, 0.178, "SHARE OF THE LAND", color="#666", fontsize=11, fontweight="bold")
order = sorted(shares, key=lambda k: -shares[k]); half = (len(order) + 1) // 2
for i, k in enumerate(order):
    col, row = divmod(i, half); x0 = 0.41 + col * 0.29; y0 = 0.145 - row * 0.029
    fig.patches.append(plt.Rectangle((x0, y0 - 0.006), 0.026, 0.02, transform=fig.transFigure, fc=COL[k], ec="#0003", lw=.5))
    fig.text(x0 + 0.036, y0 + 0.004, NAMES[k], fontsize=11.5, fontweight="bold", va="center", color="#222")
    fig.text(x0 + 0.255, y0 + 0.004, f"{shares[k]:.1f}%", fontsize=11.5, fontweight="bold", va="center", ha="right", color="#222")
fig.text(0.04, 0.015, f"Source: Esri / Impact Observatory, Sentinel-2 10 m Land Use/Land Cover {a.year}, sampled at ~200 m. "
         "Shares by true area (UTM 46N).", fontsize=7.6, style="italic", color="#666")
fig.savefig(a.out, facecolor=BG); print({NAMES[k]: round(v, 1) for k, v in shares.items()})


{'Crops': np.float64(47.8), 'Rangeland': np.float64(1.2), 'Trees': np.float64(22.3), 'Built area': np.float64(17.7), 'Bare ground': np.float64(0.8), 'Water': np.float64(9.2), 'Flooded veg.': np.float64(1.1)}
